In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
from sklearn.model_selection import train_test_split

import config
from src.db_io import leer_tabla_sqlite

df = leer_tabla_sqlite(config.ORO_DB, "cliente_features")
poblacion = pd.read_csv(config.OUTPUTS_DIR / "eda" / "poblacion_modelado.csv")
df = df[df["numero_id"].isin(poblacion["numero_id"])].reset_index(drop=True)

COLUMNAS_SENSIBLES = ["desc_genero", "grupo_edad", "desc_tipo_de_vivienda"]
COLUMNAS_FUGA = [c for c in df.columns if c.startswith("invesbot_") or c.startswith("inversion_virtual_")]
COLUMNAS_NO_FEATURE = ["numero_id", "etiqueta_adopcion", "excluir_modelado"] + COLUMNAS_SENSIBLES + COLUMNAS_FUGA

feature_cols = [c for c in df.columns if c not in COLUMNAS_NO_FEATURE]
X = pd.get_dummies(df[feature_cols], columns=["desc_segmento"], drop_first=True)
X = X.fillna({"estimador_ingreso": X["estimador_ingreso"].median()})
y = df["etiqueta_adopcion"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=config.TEST_SIZE, random_state=config.RANDOM_STATE, stratify=y
)
print(f"train: {X_train.shape}, test: {X_test.shape}, tasa adopción train: {y_train.mean():.4f}")

train: (615740, 31), test: (153935, 31), tasa adopción train: 0.0824


In [2]:
assert not any(c.startswith("invesbot_") or c.startswith("inversion_virtual_") for c in X.columns)
assert not any(c in X.columns for c in COLUMNAS_SENSIBLES)
print("OK: sin fuga de datos ni variables sensibles en las features")

OK: sin fuga de datos ni variables sensibles en las features
